## Komórka 1: Importy i Globalna Konfiguracja

In [ ]:
import os
import json
import torch
from torch.cuda import device
from torch.utils.data import DataLoader
from IPython.display import clear_output
import config
from training.pipeline import process_single_hyperparameter_run
from data_processing import preparation, batching, dataset
from vizualization import plotting
from evaluation.post_run_analysis import visualize_best_model_outputs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_HOME = config.DATA_HOME_DEFAULT
OUTPUT_BASE_DIR = config.OUTPUT_BASE_DIR_DEFAULT
SEARCH_RUN_NAME = "hyperparam_set_v1"
MAIN_SEARCH_ARTIFACTS_DIR = os.path.join(OUTPUT_BASE_DIR, SEARCH_RUN_NAME)

os.makedirs(DATA_HOME, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(MAIN_SEARCH_ARTIFACTS_DIR, exist_ok=True)

print(f"Używany DATA_HOME: {DATA_HOME}")
print(f"Używany OUTPUT_BASE_DIR: {OUTPUT_BASE_DIR}")
print(f"Używany MAIN_SEARCH_ARTIFACTS_DIR: {MAIN_SEARCH_ARTIFACTS_DIR}")

param_grid = []
try:
    with open(config.DEFAULT_HYPERPARAMETER_FILE, 'r', encoding='utf-8') as f:
        param_grid = json.load(f)
    print(f"Wczytano {len(param_grid)} zestawów hiperparametrów z: {config.DEFAULT_HYPERPARAMETER_FILE}")
    if param_grid:
        print(f"Przykładowy pierwszy wczytany zestaw: {param_grid[0]}")
except FileNotFoundError:
    print(f"BŁĄD: Plik z zestawami hiperparametrów '{config.DEFAULT_HYPERPARAMETER_FILE}' nie został znaleziony.")
except json.JSONDecodeError as e_json:
    print(f"BŁĄD: Plik '{config.DEFAULT_HYPERPARAMETER_FILE}' zawiera niepoprawny format JSON. Szczegóły: {e_json}")
except Exception as e:
    print(f"BŁĄD podczas wczytywania zestawów hiperparametrów: {e}")

if not param_grid:
    print("OSTRZEŻENIE: `param_grid` jest pusta. Przeszukiwanie może nie zostać wykonane poprawnie.")

def jupyter_notebook_clear_output(wait=True):
    if config.CLEAR_CONSOLE_EVERY_N_RUNS > 0:
        clear_output(wait=wait)
    else:
        pass


## Komórka 2: Przygotowanie podziału ID utworów i preprocessing danych

In [ ]:
print("--- Etap 1: Przygotowanie i Podział Danych ---")

track_ids_map_split = preparation.prepare_track_splits(
    data_home=DATA_HOME,
    problematic_files_list=config.PROBLEMATIC_FILES,
    test_split_fraction=config.TEST_SPLIT_SIZE,
    validation_split_fraction=config.VALIDATION_SPLIT_SIZE,
    seed=config.RANDOM_SEED,
    output_dir_for_ids=OUTPUT_BASE_DIR
)

if any(len(ids) > 0 for ids in track_ids_map_split.values()):
    print("\nUruchamianie preprocessingu GuitarSet (pominie istniejące pliki)...")
    preparation.preprocess_guitarset_data(
        guitarset_data_home=DATA_HOME,
        processed_output_base_dir=OUTPUT_BASE_DIR,
        track_ids_map=track_ids_map_split,
        audio_sample_rate=config.SAMPLE_RATE,
        audio_n_fft=config.N_FFT,
        audio_hop_length=config.HOP_LENGTH,
        audio_n_mels=config.N_MELS
    )
    print("Preprocessing zakończony lub pliki już istniały.")
else:
    print("Brak utworów do przetworzenia. Preprocessing nie został uruchomiony.")

## Komórka 3: Tworzenie obiektów Dataset i DataLoader

In [ ]:
print("\n--- Etap 2: Tworzenie Datasetów i DataLoaderów ---")

train_dataset, validation_dataset, test_dataset = None, None, None
train_dataloader, validation_dataloader, test_dataloader = None, None, None

train_dataset_constructor_params = {
    **config.DATASET_COMMON_PARAMS,
    **config.DATASET_TRAIN_AUGMENTATION_PARAMS,
    "label_transform_function": dataset.create_frame_level_labels,
    "guitarset_data_home": DATA_HOME
}

eval_dataset_constructor_params = {
    **config.DATASET_COMMON_PARAMS,
    **config.DATASET_EVAL_AUGMENTATION_PARAMS,
    "label_transform_function": dataset.create_frame_level_labels,
    "guitarset_data_home": DATA_HOME
}

if track_ids_map_split.get('train') and len(track_ids_map_split['train']) > 0:
    train_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='train',
        **train_dataset_constructor_params
    )
    if len(train_dataset) > 0:
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór treningowy: {len(train_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór treningowy jest pusty.")

if track_ids_map_split.get('validation') and len(track_ids_map_split['validation']) > 0:
    validation_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='validation',
        **eval_dataset_constructor_params
    )
    if len(validation_dataset) > 0:
        validation_dataloader = DataLoader(
            validation_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór walidacyjny: {len(validation_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór walidacyjny jest pusty.")

if track_ids_map_split.get('test') and len(track_ids_map_split['test']) > 0:
    test_dataset = dataset.GuitarSetTabDataset(
        processed_data_base_dir=OUTPUT_BASE_DIR,
        data_split_name='test',
        **eval_dataset_constructor_params
    )
    if len(test_dataset) > 0:
        test_dataloader = DataLoader(
            test_dataset,
            batch_size=config.BATCH_SIZE_DEFAULT,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            collate_fn=batching.collate_fn_pad
        )
        print(f"Utworzono zbiór testowy: {len(test_dataset)} próbek, DataLoader gotowy.")
    else:
        print("Zbiór testowy jest pusty.")

## Komórka 4: Wizualizacja Danych

In [ ]:
print("\n--- Etap 3: Wizualizacja Próbki Danych ---")

if train_dataloader and len(train_dataloader) > 0:
    try:
        features_batch, (onset_targets_batch, fret_targets_batch), lengths_batch, _raw_labels_batch = next(iter(train_dataloader))

        sample_idx_to_plot = 0
        if features_batch.numel() > 0 and sample_idx_to_plot < features_batch.shape[0]:
            direct_features_sample, (direct_onset_targets_sample, direct_fret_targets_sample), _ = train_dataset[sample_idx_to_plot]

            track_id_for_plot = f"Próbka {sample_idx_to_plot} z datasetu"
            if hasattr(train_dataset, 'base_track_ids') and len(train_dataset.base_track_ids) > sample_idx_to_plot:
                 track_id_for_plot = train_dataset.base_track_ids[sample_idx_to_plot]

            print(f"\nGenerowanie wizualizacji dla utworu (ID bazowe): {track_id_for_plot}")
            plotting.plot_sample_data_summary(
                features_tensor=direct_features_sample,
                onset_labels_tensor=direct_onset_targets_sample,
                fret_labels_tensor=direct_fret_targets_sample,
                sampling_rate=config.SAMPLE_RATE,
                hop_len=config.HOP_LENGTH,
                track_id_display=track_id_for_plot,
                max_frets_visual=config.MAX_FRETS,
                num_strings_visual=config.DEFAULT_NUM_STRINGS
            )
        else:
            print(f"Batch jest pusty lub indeks {sample_idx_to_plot} jest nieprawidłowy dla rozmiaru batcha {features_batch.shape[0]}.")

    except StopIteration:
        print("Train DataLoader jest pusty. Nie można zwizualizować próbki.")
    except IndexError as e_idx:
        print(f"Błąd indeksowania podczas próby wizualizacji: {e_idx}")
    except Exception as e:
        print(f"Wystąpił błąd podczas wizualizacji danych: {e}")
        import traceback
        traceback.print_exc()

else:
    print("Train DataLoader nie jest dostępny lub jest pusty. Pomijam wizualizację.")

## Komórka 5: Trening i wizualizacja treningu

In [ ]:
print(f"\n{'=' * 20} ROZPOCZYNANIE PRZESZUKIWANIA HIPERPARAMETRÓW {'=' * 20}")

all_new_run_summaries_this_session = []
MAIN_SEARCH_ARTIFACTS_DIR = r"C:\Users\lukig\Documents\Programming\music-transcription\python\results\hyperparam_search"
current_active_augmentation_params = config.DATASET_TRAIN_AUGMENTATION_PARAMS

os.makedirs(MAIN_SEARCH_ARTIFACTS_DIR, exist_ok=True)
print(f"Główny katalog artefaktów przeszukiwania: {MAIN_SEARCH_ARTIFACTS_DIR}")

if not param_grid:
    print("BŁĄD KRYTYCZNY: `param_grid` jest pusta. Sprawdź wczytywanie pliku JSON z hiperparametrami.")
else:
    highest_existing_run_number = 0
    if os.path.exists(MAIN_SEARCH_ARTIFACTS_DIR):
        for item in os.listdir(MAIN_SEARCH_ARTIFACTS_DIR):
            if os.path.isdir(os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, item)) and item.startswith("run_"):
                try:
                    num_str = item.split('_')[1]
                    run_num = int(num_str)
                    if run_num > highest_existing_run_number:
                        highest_existing_run_number = run_num
                except (IndexError, ValueError):
                    print(f"Ostrzeżenie: Nie udało się sparsować numeru przebiegu z folderu: {item}")
                    continue
    next_run_start_number = highest_existing_run_number + 1
    print(f"Nowe przebiegi rozpoczną się od numeru (ID): {next_run_start_number}")

    total_new_runs = len(param_grid)
    for run_idx_in_current_batch, hyperparams_combo_current in enumerate(param_grid):
        current_run_global_id = next_run_start_number + run_idx_in_current_batch

        if run_idx_in_current_batch > 0 and \
           hasattr(config, 'CLEAR_CONSOLE_EVERY_N_RUNS') and \
           config.CLEAR_CONSOLE_EVERY_N_RUNS > 0 and \
           run_idx_in_current_batch % config.CLEAR_CONSOLE_EVERY_N_RUNS == 0:
            if callable(jupyter_notebook_clear_output):
                jupyter_notebook_clear_output(wait=True)
                print("Output konsoli wyczyszczony. Kontynuacja przeszukiwania...")

        print(f"\n\n{'=' * 30} ROZPOCZYNANIE PRZEBIEGU {run_idx_in_current_batch + 1}/{total_new_runs} (Globalne ID: {current_run_global_id}) {'=' * 30}")
        print(f"Używane hiperparametry: {hyperparams_combo_current}")

        run_summary = process_single_hyperparameter_run(
            run_id=current_run_global_id,
            hyperparams_combo=hyperparams_combo_current,
            current_augmentation_params=current_active_augmentation_params,
            config_obj=config,
            main_artifacts_dir=MAIN_SEARCH_ARTIFACTS_DIR,
            train_loader=train_dataloader,
            validation_loader=validation_dataloader,
            test_loader=test_dataloader,
            jupyter_notebook_clear_output_func=jupyter_notebook_clear_output
        )

        all_new_run_summaries_this_session.append(run_summary)

        summary_path_raw = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, "hyperparameter_search_summary.jsonl")
        with open(summary_path_raw, "a", encoding="utf-8") as f_sum_raw:
            f_sum_raw.write(json.dumps(run_summary) + "\n")

        print(f"\n{'=' * 30} ZAKOŃCZONO PRZEBIEG {run_idx_in_current_batch + 1}/{total_new_runs} (Globalne ID: {current_run_global_id}) - Status: {run_summary.get('status', 'UNKNOWN_STATUS')} {'=' * 30}\n")

    print("\n\n--- Podsumowanie Całego Przeszukiwania Hiperparametrów (wszystkie zapisane przebiegi) ---")
    
    all_runs_from_file = []
    summary_path_raw_for_final_sort = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, "hyperparameter_search_summary.jsonl")
    
    if os.path.exists(summary_path_raw_for_final_sort):
        with open(summary_path_raw_for_final_sort, "r", encoding="utf-8") as f_sum_all:
            for line_idx, line in enumerate(f_sum_all):
                try:
                    all_runs_from_file.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Ostrzeżenie: Pomięto błędną linię ({line_idx+1}) w pliku podsumowania '{summary_path_raw_for_final_sort}': {e}. Treść: '{line.strip()}'")
    else:
        print(f"Ostrzeżenie: Plik podsumowania '{summary_path_raw_for_final_sort}' nie istnieje. Nie można wygenerować posortowanego podsumowania.")

    if all_runs_from_file:
        sort_key_primary_metric = f"best_{config.CHECKPOINT_METRIC_DEFAULT}"

        def get_sortable_value(result_dict):
            primary_metric_val = -float('inf') if "loss" not in sort_key_primary_metric.lower() else float('inf')
            raw_metric_val = result_dict.get(sort_key_primary_metric)
            if isinstance(raw_metric_val, (int, float)):
                primary_metric_val = raw_metric_val
            
            secondary_metric_val = -float('inf')
            test_metrics_dict = result_dict.get("test_metrics")
            if isinstance(test_metrics_dict, dict):
                raw_secondary_val = test_metrics_dict.get("test_tdr_f1", -float('inf'))
                if isinstance(raw_secondary_val, (int, float)):
                    secondary_metric_val = raw_secondary_val
            
            if "loss" in sort_key_primary_metric.lower():
                primary_metric_val *= -1 
            
            return (primary_metric_val, secondary_metric_val)

        completed_runs = [res for res in all_runs_from_file if res.get("status") == "COMPLETED"]
        error_runs = [res for res in all_runs_from_file if res.get("status") != "COMPLETED"]

        sorted_completed_runs = sorted(
            completed_runs,
            key=get_sortable_value,
            reverse=True
        )
        final_sorted_list_for_summary_file = sorted_completed_runs + error_runs

        summary_file_path_final_sorted = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, "hyperparameter_search_summary_sorted.jsonl")
        with open(summary_file_path_final_sorted, "w", encoding="utf-8") as f_final_sum:
            for res_item in final_sorted_list_for_summary_file:
                f_final_sum.write(json.dumps(res_item) + "\n")
        print(f"Zapisano posortowane zbiorcze podsumowanie (uwzględniając wszystkie przebiegi) do: {summary_file_path_final_sorted}")

        print("\nNajlepsze przebiegi (Top 5 lub wszystkie, jeśli mniej niż 5 ukończonych), następnie błędy:")
        num_top_to_show = min(5, len(sorted_completed_runs))

        for i, result_item_summary in enumerate(final_sorted_list_for_summary_file):
            is_top_run = result_item_summary.get("status") == "COMPLETED" and (sorted_completed_runs.index(result_item_summary) < num_top_to_show if result_item_summary in sorted_completed_runs else False)
            
            if is_top_run:
                print(f"\nPozycja {sorted_completed_runs.index(result_item_summary) + 1}: Przebieg {result_item_summary.get('run_index', 'N/A')} (Folder: {result_item_summary.get('run_folder_name', 'N/A')})")
                print(f"  Parametry: {result_item_summary.get('params_combo', {})}")
                raw_best_val_metric = result_item_summary.get(sort_key_primary_metric, "N/A")
                val_metric_str = f"{raw_best_val_metric:.4f}" if isinstance(raw_best_val_metric, float) else str(raw_best_val_metric)
                print(f"  Najlepsza metryka walidacyjna ({config.CHECKPOINT_METRIC_DEFAULT}): {val_metric_str}")

                test_metrics_data_dict = result_item_summary.get('test_metrics', "N/A")
                if isinstance(test_metrics_data_dict, dict):
                    onset_f1 = test_metrics_data_dict.get('test_onset_f1_mir_eval', 0)
                    onset_p = test_metrics_data_dict.get('test_onset_precision_mir_eval', 0)
                    onset_r = test_metrics_data_dict.get('test_onset_recall_mir_eval', 0)
                    ftab_f1 = test_metrics_data_dict.get('test_ftab_frame', 0)
                    tdr_r = test_metrics_data_dict.get('test_tdr_recall_note', 0) 
                    tdr_p = test_metrics_data_dict.get('test_tdr_precision_note', 0)
                    tdr_f1 = test_metrics_data_dict.get('test_tdr_f1_note', 0) 

                    print(f"  Test Onset F1 (mir_eval): {onset_f1:.4f} (P: {onset_p:.4f}, R: {onset_r:.4f})")
                    print(f"  Test FTab (ramkowy): {ftab_f1:.4f}")
                    print(f"  Test TDR F1 (nutowy): {tdr_f1:.4f} (P: {tdr_p:.4f}, R: {tdr_r:.4f})")
                else:
                    print(f"  Metryki testowe: {test_metrics_data_dict}")
                print(f"  Zakończono na epoce: {result_item_summary.get('stopped_epoch', 0)}")
                print(f"  Czas trwania: {result_item_summary.get('run_duration_minutes', 0):.2f} min")
            
            elif result_item_summary.get("status") != "COMPLETED":
                if i == 0 or (i > 0 and final_sorted_list_for_summary_file[i-1].get("status") == "COMPLETED"):
                     print("\n--- Przebiegi zakończone błędem ---")
                print(f"\nPrzebieg z BŁĘDEM: {result_item_summary.get('run_index', 'N/A')} (Folder: {result_item_summary.get('run_folder_name', 'N/A')})")
                print(f"  Status: {result_item_summary.get('status', 'N/A')}")
                print(f"  Parametry: {result_item_summary.get('params_combo', {})}")
                print(f"  Błąd: {result_item_summary.get('error', 'Brak szczegółów błędu')}")
    else:
        print("Nie przeprowadzono żadnych przebiegów lub nie udało się wczytać pliku podsumowania (np. jeśli `param_grid` była pusta lub plik jsonl jest uszkodzony).")

## Komórka 6: Generowanie wyników MIDI oraz Tabulatury

In [ ]:
MAIN_SEARCH_ARTIFACTS_DIR = r"C:\Users\lukig\Documents\Programming\music-transcription\python\results\hyperparam_search"

print(f"Używany katalog artefaktów przeszukiwania: {MAIN_SEARCH_ARTIFACTS_DIR}")

if 'test_dataset' not in locals() or test_dataset is None:
    print("BŁĄD KRYTYCZNY: Zmienna 'test_dataset' nie jest zdefiniowana lub jest None.")
    print("Proszę najpierw załadować zbiór testowy.")
else:
    print(f"Zbiór testowy ('test_dataset') jest dostępny i zawiera {len(test_dataset)} próbek.")
    print("\n--- Dostępne foldery przebiegów w katalogu artefaktów ---")
    available_run_folders = []
    if os.path.exists(MAIN_SEARCH_ARTIFACTS_DIR):
        for item in os.listdir(MAIN_SEARCH_ARTIFACTS_DIR):
            potential_run_path = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, item)
            if os.path.isdir(potential_run_path) and item.startswith("run_") and os.path.exists(os.path.join(potential_run_path, "best_model.pth")):
                available_run_folders.append(item)
    
    if not available_run_folders:
        print(f"Brak folderów przebiegów (zaczynających się od 'run_' i zawierających 'best_model.pth') w {MAIN_SEARCH_ARTIFACTS_DIR}")
    else:
        print("Znaleziono następujące foldery przebiegów:")
        for idx, folder_name in enumerate(available_run_folders):
            print(f"[{idx + 1}] {folder_name}")

        selected_run_folder_name = None
        while True:
            try:
                choice_str = input(f"Wybierz numer folderu przebiegu do wizualizacji (1-{len(available_run_folders)}) lub wpisz '0' aby pominąć: ")
                choice = int(choice_str)
                if 0 <= choice <= len(available_run_folders):
                    if choice > 0:
                        selected_run_folder_name = available_run_folders[choice - 1]
                    break
                else:
                    print(f"Nieprawidłowy wybór. Podaj liczbę od 0 do {len(available_run_folders)}.")
            except ValueError:
                print("Nieprawidłowe dane wejściowe. Podaj liczbę.")

        if selected_run_folder_name:
            print(f"\nWybrano do wizualizacji przebieg z folderu: {selected_run_folder_name}")
            selected_run_summary_for_viz = None
            summary_file_path = os.path.join(MAIN_SEARCH_ARTIFACTS_DIR, "hyperparameter_search_summary_sorted.jsonl")
            
            if os.path.exists(summary_file_path):
                with open(summary_file_path, "r", encoding="utf-8") as f_sum:
                    for line in f_sum:
                        try:
                            run_data = json.loads(line)
                            if run_data.get("run_folder_name") == selected_run_folder_name:
                                selected_run_summary_for_viz = run_data
                                print(f"Znaleziono podsumowanie dla przebiegu: {selected_run_folder_name}")
                                break
                        except json.JSONDecodeError:
                            print(f"Ostrzeżenie: Pomięto błędną linię w pliku {summary_file_path}")
                            continue
            
            if not selected_run_summary_for_viz:
                print(f"OSTRZEŻENIE: Nie udało się znaleźć podsumowania dla przebiegu '{selected_run_folder_name}' w pliku '{summary_file_path}'.")
                print("Funkcja wizualizacji może nie zadziałać poprawnie bez 'params_combo' i 'optimal_threshold_at_best_val_metric_frame'.")
                print("Rozważ ręczne utworzenie słownika 'selected_run_summary_for_viz' z wymaganymi kluczami, jeśli znasz parametry.")
            if selected_run_summary_for_viz:
                visualize_best_model_outputs(
                    sorted_completed_runs_list=[selected_run_summary_for_viz],
                    main_search_artifacts_dir_path=MAIN_SEARCH_ARTIFACTS_DIR,
                    config_obj=config, 
                    test_dataset_instance=test_dataset,
                    current_device=device,
                    data_home_path=DATA_HOME 
                )
            else:
                print(f"Nie można kontynuować wizualizacji bez podsumowania dla przebiegu '{selected_run_folder_name}'.")
        else:
            print("Pominięto wizualizację na życzenie użytkownika.")